# Gold governance hardening (Step 8 follow-up)

Applies the user-approved fix for the gap found in Step 8: `txn_features_gold` and `scored_transactions` had no row filter, and `customer_risk_profile_gold.avg_amount_30d` averages unfiltered transactions, while `account users` holds ALL PRIVILEGES on `workshop_guy_catalog`.

* Row filter `filter_high_amount_txns` on `amount` of `txn_features_gold` (SDP definition in `gold_feature_layer.sql`) and `scored_transactions` (`ALTER TABLE`).
* New mask `mask_avg_amount_30d` on `customer_risk_profile_gold.avg_amount_30d` (SDP definition).
* Then one run of job `fe_bar_gold_to_lakebase` proves the pipeline, the scoring write and the Lakebase sync still work, and before/after counts are compared.

In [0]:
import datetime, time
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
FQ = "workshop_guy_catalog.fe_bar_a0"
JOB_ID = 1115396029932218
GOLD = ["txn_features_gold", "customer_risk_profile_gold", "scored_transactions"]
SYNCED = ["fe_bar_lakebase.fraud.txn_features", "fe_bar_lakebase.fraud.customer_risk_profile",
          "fe_bar_lakebase.fraud.txn_risk_current", "fe_bar_lakebase.fraud.risk_score_calibration"]
DASH = [f"fe_bar_lakebase.fraud_dash.{v}" for v in ["case_summary", "case_actions_daily", "score_distribution", "exposure", "baseline_comparison", "triage_queue"]]
print("run_at_utc:", datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"))
print("current_user:", spark.sql("SELECT current_user()").first()[0], "| pii member:", spark.sql("SELECT is_account_group_member('pii')").first()[0])

def governance(t):
    """Row filter and column masks from information_schema (works for tables and MVs)."""
    rf = spark.sql(f"SELECT filter_name, target_columns FROM workshop_guy_catalog.information_schema.row_filters WHERE table_schema='fe_bar_a0' AND table_name='{t}'").collect()
    cm = spark.sql(f"SELECT column_name, mask_name FROM workshop_guy_catalog.information_schema.column_masks WHERE table_schema='fe_bar_a0' AND table_name='{t}'").collect()
    return ([f"{r.filter_name}({r.target_columns})" for r in rf] or None, [f"{r.column_name}->{r.mask_name}" for r in cm] or None)

def state():
    s = {}
    for t in GOLD:
        cols = [c.name for c in spark.table(f"{FQ}.{t}").schema]
        n = spark.table(f"{FQ}.{t}").count()
        gt = spark.table(f"{FQ}.{t}").filter("amount > 10000").count() if "amount" in cols else None
        extra = spark.sql(f"SELECT round(sum(avg_amount_30d), 2) FROM {FQ}.{t}").first()[0] if "avg_amount_30d" in cols else None
        s[t] = (n, gt, extra, *governance(t))
    for t in SYNCED + DASH:
        s[t] = (spark.table(t).count(), None, None, None, None)
    return s

def show(s):
    print(f"{'object':48} {'rows':>9} {'amount>10000':>13} {'sum(avg_amount_30d)':>20}  row_filter / masks")
    for k, (n, gt, ex, rf, cm) in s.items():
        print(f"{k.replace('workshop_guy_catalog.',''):48} {n:>9,} {'' if gt is None else gt:>13} {'' if ex is None else ex:>20}  {rf or ''} {cm or ''}")

BEFORE = state()
show(BEFORE)

run_at_utc: 2026-11-04T00:40:12+00:00
current_user: alex.owen@databricks.com | pii member: True
object                                                rows  amount>10000  sum(avg_amount_30d)  row_filter / masks
txn_features_gold                                  464,632           751                        
customer_risk_profile_gold                           9,289                         2010281.99   
scored_transactions                                464,632           751                        
fe_bar_lakebase.fraud.txn_features                 464,632                                      
fe_bar_lakebase.fraud.customer_risk_profile          9,289                                      
fe_bar_lakebase.fraud.txn_risk_current             464,632                                      
fe_bar_lakebase.fraud.risk_score_calibration            12                                      
fe_bar_lakebase.fraud_dash.case_summary                221                                      
fe_bar_lakebas

In [0]:
# Same pattern as the existing mask_avg_monthly_spend: pii members see the value, everyone else NULL.
spark.sql(f"""
CREATE FUNCTION IF NOT EXISTS {FQ}.mask_avg_amount_30d(avg_amount_30d DOUBLE)
RETURNS DOUBLE
COMMENT 'Column mask for customer_risk_profile_gold.avg_amount_30d (averages unfiltered txns incl. amount > 10000): pii sees the value, others NULL'
RETURN IF(IS_ACCOUNT_GROUP_MEMBER('pii'), avg_amount_30d, NULL)""")
# scored_transactions is a managed Delta table rewritten with INSERT OVERWRITE, which keeps table-level filters.
spark.sql(f"ALTER TABLE {FQ}.scored_transactions SET ROW FILTER {FQ}.filter_high_amount_txns ON (amount)")
for f in ["filter_high_amount_txns", "mask_avg_amount_30d"]:
    body = [r[0] for r in spark.sql(f"DESCRIBE FUNCTION EXTENDED {FQ}.{f}").collect() if r[0].startswith("Body")]
    print(f, "->", body[0] if body else "?")
print("scored_transactions:", governance("scored_transactions"))
# txn_features_gold and customer_risk_profile_gold are SDP materialized views: their filter/mask live in
# gold_feature_layer.sql and take effect on the next pipeline update (cell 3).
print(open("/Workspace/Users/alex.owen@databricks.com/FE-BAR-2026-repo/gold_feature_layer.sql").read().count("filter_high_amount_txns"), "row filter clause(s) and",
      open("/Workspace/Users/alex.owen@databricks.com/FE-BAR-2026-repo/gold_feature_layer.sql").read().count("MASK workshop_guy_catalog"), "mask clause(s) in gold_feature_layer.sql")

filter_high_amount_txns -> Body:          IF(IS_ACCOUNT_GROUP_MEMBER('pii'), TRUE, amount <= 10000)
mask_avg_amount_30d -> Body:          IF(IS_ACCOUNT_GROUP_MEMBER('pii'), avg_amount_30d, NULL)
scored_transactions: (['workshop_guy_catalog.fe_bar_a0.filter_high_amount_txns(amount)'], None)
1 row filter clause(s) and 1 mask clause(s) in gold_feature_layer.sql


In [0]:
# This cell originally started the job with w.jobs.run_now(JOB_ID) at 2026-11-04 ~00:41 UTC (run 273696236528085)
# and asserted SUCCESS. The run FAILED at lakebase_sync_refresh, so the cell raised. It now records that run
# read-only (no new job run) so the committed notebook holds the evidence as text, not as an error output.
import datetime
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
JOB_ID, RUN_ID = 1115396029932218, 273696236528085
r = w.jobs.get_run(RUN_ID)
ts = lambda ms: datetime.datetime.fromtimestamp(ms / 1000, datetime.timezone.utc).isoformat(timespec="seconds") if ms else None
print(f"job {JOB_ID} run {RUN_ID}: start {ts(r.start_time)} end {ts(r.end_time)} -> {r.state.life_cycle_state.value} / {r.state.result_state.value}")
print("  message:", r.state.state_message)
for t in sorted(r.tasks, key=lambda t: t.start_time or 0):
    print(f"  task {t.task_key:24} {t.state.result_state.value if t.state.result_state else t.state.life_cycle_state.value:10} {ts(t.start_time)} -> {ts(t.end_time)}")
SYNC_PID = "f8690359-4bb1-45b9-8fdb-5bd892239db2"
for e in w.pipelines.list_pipeline_events(SYNC_PID, max_results=200):
    if e.origin and e.origin.update_id and e.origin.update_id.startswith("3eb6c6df") and e.event_type == "update_progress" and e.level and e.level.value == "ERROR":
        print("  sync pipeline update 3eb6c6df error:", (e.error.exceptions[0].message if e.error and e.error.exceptions else e.message)[:600])
        break
print("Result: gold refresh (row filter + mask applied) and batch scoring succeeded; the Lakebase sync could not read")
print("scored_transactions with a table-level row filter (synced tables read a pinned version = time travel). See cell 3b.")

job 1115396029932218 run 273696236528085: start 2026-11-04T00:41:28+00:00 end 2026-11-04T00:46:06+00:00 -> INTERNAL_ERROR / FAILED
  message: Task lakebase_sync_refresh failed with message: Workload failed, see run output for details. This caused all downstream tasks to get skipped.
  task gold_pipeline_refresh    SUCCESS    2026-11-04T00:41:28+00:00 -> 2026-11-04T00:42:51+00:00
  task batch_scoring            SUCCESS    2026-11-04T00:42:52+00:00 -> 2026-11-04T00:45:23+00:00
  task lakebase_sync_refresh    FAILED     2026-11-04T00:45:24+00:00 -> 2026-11-04T00:46:06+00:00
  sync pipeline update 3eb6c6df error: [SYNCED_TABLE_USER_ERROR.SOURCE_READ_ERROR] Synced table pipeline user error. Failed to read from the source table workshop_guy_catalog.fe_bar_a0.scored_transactions. Please verify if the source table can be read successfully through a SELECT query.
Detail: [ROW_LEVEL_SECURITY_FEATURE_NOT_SUPPORTED.TIME_TRAVEL] Row level security policies for `workshop_guy_catalog`.`fe_bar_a0`.`sc

In [0]:
# Cell 3 failed at lakebase_sync_refresh (run 273696236528085):
#   SYNCED_TABLE_USER_ERROR.SOURCE_READ_ERROR on scored_transactions
#   ROW_LEVEL_SECURITY_FEATURE_NOT_SUPPORTED.TIME_TRAVEL (the sync reads a pinned table version).
# Gold refresh and batch scoring succeeded. The user chose option B: remove only the
# scored_transactions filter and re-test whether the filtered/masked gold MVs sync.
# The Python session was reset after cell 3, so this cell re-declares what it and cell 4 need
# (re-running cell 1 would overwrite its recorded BEFORE output).
import datetime, time
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
FQ = "workshop_guy_catalog.fe_bar_a0"
RUN_ID = 273696236528085
GOLD = ["txn_features_gold", "customer_risk_profile_gold", "scored_transactions"]
SYNCED = ["fe_bar_lakebase.fraud.txn_features", "fe_bar_lakebase.fraud.customer_risk_profile",
          "fe_bar_lakebase.fraud.txn_risk_current", "fe_bar_lakebase.fraud.risk_score_calibration"]
DASH = [f"fe_bar_lakebase.fraud_dash.{v}" for v in ["case_summary", "case_actions_daily", "score_distribution", "exposure", "baseline_comparison", "triage_queue"]]

def governance(t):
    rf = spark.sql(f"SELECT filter_name, target_columns FROM workshop_guy_catalog.information_schema.row_filters WHERE table_schema='fe_bar_a0' AND table_name='{t}'").collect()
    cm = spark.sql(f"SELECT column_name, mask_name FROM workshop_guy_catalog.information_schema.column_masks WHERE table_schema='fe_bar_a0' AND table_name='{t}'").collect()
    return ([f"{r.filter_name}({r.target_columns})" for r in rf] or None, [f"{r.column_name}->{r.mask_name}" for r in cm] or None)

def state():
    s = {}
    for t in GOLD:
        cols = [c.name for c in spark.table(f"{FQ}.{t}").schema]
        n = spark.table(f"{FQ}.{t}").count()
        gt = spark.table(f"{FQ}.{t}").filter("amount > 10000").count() if "amount" in cols else None
        extra = spark.sql(f"SELECT round(sum(avg_amount_30d), 2) FROM {FQ}.{t}").first()[0] if "avg_amount_30d" in cols else None
        s[t] = (n, gt, extra, *governance(t))
    for t in SYNCED + DASH:
        s[t] = (spark.table(t).count(), None, None, None, None)
    return s

def show(s):
    print(f"{'object':48} {'rows':>9} {'amount>10000':>13} {'sum(avg_amount_30d)':>20}  row_filter / masks")
    for k, (n, gt, ex, rf, cm) in s.items():
        print(f"{k.replace('workshop_guy_catalog.',''):48} {n:>9,} {'' if gt is None else gt:>13} {'' if ex is None else ex:>20}  {rf or ''} {cm or ''}")

r = w.jobs.get_run(RUN_ID)
print("job run", RUN_ID, r.state.result_state.value, "|", r.state.state_message)
for t in r.tasks:
    print(f"  task {t.task_key:24} {t.state.result_state.value if t.state.result_state else t.state.life_cycle_state.value}")

spark.sql(f"ALTER TABLE {FQ}.scored_transactions DROP ROW FILTER")
print("scored_transactions governance now:", governance("scored_transactions"))

SYNC_PID = "f8690359-4bb1-45b9-8fdb-5bd892239db2"
u = w.pipelines.start_update(SYNC_PID)
print("sync pipeline update", u.update_id)
t0 = time.time()
while True:
    st = w.pipelines.get_update(SYNC_PID, u.update_id).update.state.value
    if st in ("COMPLETED", "FAILED", "CANCELED"):
        break
    time.sleep(20)
print(f"sync update state: {st} after {int(time.time()-t0)} s")
for e in w.pipelines.list_pipeline_events(SYNC_PID, max_results=50):
    if e.origin and e.origin.update_id == u.update_id and e.level and e.level.value == "ERROR":
        msg = e.error.exceptions[0].message if e.error and e.error.exceptions else e.message
        print("  ERROR", e.event_type, (msg or "")[:600])
for n in SYNCED:
    ds = w.database.get_synced_database_table(n).data_synchronization_status
    print(f"  {n:48} {ds.detailed_state.value}")

job run 273696236528085 FAILED | Task lakebase_sync_refresh failed with message: Workload failed, see run output for details. This caused all downstream tasks to get skipped.
  task lakebase_sync_refresh    FAILED
  task batch_scoring            SUCCESS
  task gold_pipeline_refresh    SUCCESS
scored_transactions governance now: (None, None)
sync pipeline update d1502c56-908a-4cb5-b28b-60cf0ccf91d3
sync update state: COMPLETED after 60 s
  fe_bar_lakebase.fraud.txn_features               SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE
  fe_bar_lakebase.fraud.customer_risk_profile      SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE
  fe_bar_lakebase.fraud.txn_risk_current           SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE
  fe_bar_lakebase.fraud.risk_score_calibration     SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE


In [0]:
# BEFORE as printed by cell 1 at 2026-11-04T00:40:12Z (the session was reset since; values copied from that output)
BEFORE = {
  "txn_features_gold": (464632, 751, None, None, None), "customer_risk_profile_gold": (9289, None, 2010281.99, None, None),
  "scored_transactions": (464632, 751, None, None, None),
  "fe_bar_lakebase.fraud.txn_features": (464632,)*1 + (None,)*4, "fe_bar_lakebase.fraud.customer_risk_profile": (9289,) + (None,)*4,
  "fe_bar_lakebase.fraud.txn_risk_current": (464632,) + (None,)*4, "fe_bar_lakebase.fraud.risk_score_calibration": (12,) + (None,)*4,
  "fe_bar_lakebase.fraud_dash.case_summary": (221,) + (None,)*4, "fe_bar_lakebase.fraud_dash.case_actions_daily": (8,) + (None,)*4,
  "fe_bar_lakebase.fraud_dash.score_distribution": (31,) + (None,)*4, "fe_bar_lakebase.fraud_dash.exposure": (6,) + (None,)*4,
  "fe_bar_lakebase.fraud_dash.baseline_comparison": (2,) + (None,)*4, "fe_bar_lakebase.fraud_dash.triage_queue": (24201,) + (None,)*4,
}
AFTER = state()
show(AFTER)
sync_states = {n: w.database.get_synced_database_table(n).data_synchronization_status.detailed_state.value for n in SYNCED}
print()
checks = []
for k in BEFORE:
    checks.append((f"{k}: row count unchanged for a pii reader", BEFORE[k][0] == AFTER[k][0], f"{BEFORE[k][0]:,} -> {AFTER[k][0]:,}"))
checks += [
  ("txn_features_gold has row filter filter_high_amount_txns(amount)", bool(AFTER["txn_features_gold"][3]) and "filter_high_amount_txns" in str(AFTER["txn_features_gold"][3]), AFTER["txn_features_gold"][3]),
  ("scored_transactions has NO row filter (option B: synced-table time travel is not supported with a table row filter)", AFTER["scored_transactions"][3] is None, AFTER["scored_transactions"][3]),
  ("customer_risk_profile_gold.avg_amount_30d masked", "avg_amount_30d" in str(AFTER["customer_risk_profile_gold"][4]), AFTER["customer_risk_profile_gold"][4]),
  ("pii reader still sees amount > 10000 (751) in txn_features_gold", AFTER["txn_features_gold"][1] == BEFORE["txn_features_gold"][1], AFTER["txn_features_gold"][1]),
  ("pii reader still sees amount > 10000 (751) in scored_transactions", AFTER["scored_transactions"][1] == BEFORE["scored_transactions"][1], AFTER["scored_transactions"][1]),
  ("pii reader sum(avg_amount_30d) unchanged", AFTER["customer_risk_profile_gold"][2] == BEFORE["customer_risk_profile_gold"][2], AFTER["customer_risk_profile_gold"][2]),
]
for n, s in sync_states.items():
    checks.append((f"{n} synced table online and last sync succeeded", s in ("SYNCED_TABLE_ONLINE", "SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE", "SYNCED_TABLE_ONLINE_TRIGGERED_UPDATE"), s))
# The sync runs as the pipeline owner (a pii member), so Lakebase must still hold full values for fraud_gov to govern.
lb_gt = spark.sql("SELECT count(*) FROM fe_bar_lakebase.fraud.txn_features WHERE amount > 10000").first()[0]
lb_avg = spark.sql("SELECT count(avg_amount_30d), round(sum(avg_amount_30d), 2) FROM fe_bar_lakebase.fraud.customer_risk_profile").first()
checks.append(("Lakebase fraud.txn_features still has the 751 amount > 10000 rows (sync identity is pii)", lb_gt == 751, lb_gt))
checks.append(("Lakebase fraud.customer_risk_profile.avg_amount_30d unmasked (9,289 non-null, same sum)", lb_avg[0] == 9289 and lb_avg[1] == BEFORE["customer_risk_profile_gold"][2], tuple(lb_avg)))
# What a non-pii reader gets: the non-pii branch of each function, evaluated explicitly (this user is in pii).
non_pii = {
  "txn_features_gold": spark.sql(f"SELECT count(*) FROM {FQ}.txn_features_gold WHERE amount <= 10000").first()[0],
  "scored_transactions": spark.sql(f"SELECT count(*) FROM {FQ}.scored_transactions WHERE amount <= 10000").first()[0],
}
for t, n in non_pii.items():
    print(f"non-pii branch (amount <= 10000) would return {n:,} of {AFTER[t][0]:,} rows of {t}")
print("non-pii branch of mask_avg_amount_30d returns NULL for every row")
print()
for name, ok, val in checks:
    print(f"{'PASS' if ok else 'FAIL'}  {name}  [{val}]")
print(f"\n{sum(ok for _, ok, _ in checks)}/{len(checks)} checks pass")

object                                                rows  amount>10000  sum(avg_amount_30d)  row_filter / masks
txn_features_gold                                  464,632           751                       ['workshop_guy_catalog.fe_bar_a0.filter_high_amount_txns(amount)'] 
customer_risk_profile_gold                           9,289                         2010281.99   ['avg_amount_30d->workshop_guy_catalog.fe_bar_a0.mask_avg_amount_30d']
scored_transactions                                464,632           751                        
fe_bar_lakebase.fraud.txn_features                 464,632                                      
fe_bar_lakebase.fraud.customer_risk_profile          9,289                                      
fe_bar_lakebase.fraud.txn_risk_current             464,632                                      
fe_bar_lakebase.fraud.risk_score_calibration            12                                      
fe_bar_lakebase.fraud_dash.case_summary                221             

In [0]:
# Read-only. The grants were run by alex.owen@databricks.com from SQL generated in conversation 2326365276521178
# (USE CATALOG fe_bar_lakebase, USE SCHEMA fraud_dash, SELECT on the six fraud_dash views). This cell only verifies.
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
DASH = [f"fe_bar_lakebase.fraud_dash.{v}" for v in ["case_summary", "case_actions_daily", "score_distribution", "exposure", "baseline_comparison", "triage_queue"]]
SPACE_ID = "01f1d7b464d510b9aa6cde48300787ca"
GRP = "non_pii"
g = [x for x in w.groups.list(filter=f'displayName eq "{GRP}"', attributes="displayName,meta,members")][0]
pii = [x for x in w.groups.list(filter='displayName eq "pii"', attributes="displayName,members")][0]
overlap = {m.value for m in (g.members or [])} & {m.value for m in (pii.members or [])}
print(f"group {GRP}: id {g.id}, type {g.meta.resource_type} (account group), members {len(g.members or [])}, overlap with pii: {overlap or 'none'}")
assert g.meta.resource_type == "Group" and not overlap

print("members:", [m.display for m in (g.members or [])])
# risk_score_calibration (the 7th source) is already readable: account users hold ALL PRIVILEGES on workshop_guy_catalog.
print("\nUC grants to non_pii (direct; READ METADATA is inherited from account users):")
for obj in ["CATALOG fe_bar_lakebase", "SCHEMA fe_bar_lakebase.fraud_dash"] + [f"TABLE {v}" for v in DASH] + ["TABLE fe_bar_lakebase.fraud.txn_features", "TABLE fe_bar_lakebase.fraud_gov.txn_features"]:
    try:
        rows = [r.ActionType for r in spark.sql(f"SHOW GRANTS `{GRP}` ON {obj}").collect() if r.ObjectKey.replace('`','') == obj.split(' ', 1)[1]]
    except Exception as e:
        rows = [f"n/a ({type(e).__name__})"]
    print(f"  {obj:56} {rows or 'none'}")
print("\nGenie space permissions:")
for a in w.permissions.get("genie", SPACE_ID).access_control_list:
    print(f"  {(a.group_name or a.user_name or a.service_principal_name):28} {[(p.permission_level.value, 'inherited' if p.inherited else 'direct') for p in a.all_permissions]}")
has_run = any(a.group_name == GRP for a in w.permissions.get("genie", SPACE_ID).access_control_list)
print(f"non_pii CAN_RUN on the space: {'yes' if has_run else 'NOT YET (add via Share > Can Run)'}")
print("\nWarehouse 9bbc027402111984:", [(a.group_name or a.user_name or a.service_principal_name, [p.permission_level.value for p in a.all_permissions]) for a in w.permissions.get('warehouses', '9bbc027402111984').access_control_list])
print("(members of non_pii also need to be workspace users to use the warehouse via 'users' CAN_USE)")

group non_pii: id 81258610540604, type Group (account group), members 1, overlap with pii: none
members: ['Connor Hauenstein']

UC grants to non_pii (direct; READ METADATA is inherited from account users):
  CATALOG fe_bar_lakebase                                  ['USE CATALOG']
  SCHEMA fe_bar_lakebase.fraud_dash                        ['USE SCHEMA']
  TABLE fe_bar_lakebase.fraud_dash.case_summary            ['SELECT']
  TABLE fe_bar_lakebase.fraud_dash.case_actions_daily      ['SELECT']
  TABLE fe_bar_lakebase.fraud_dash.score_distribution      ['SELECT']
  TABLE fe_bar_lakebase.fraud_dash.exposure                ['SELECT']
  TABLE fe_bar_lakebase.fraud_dash.baseline_comparison     ['SELECT']
  TABLE fe_bar_lakebase.fraud_dash.triage_queue            ['SELECT']
  TABLE fe_bar_lakebase.fraud.txn_features                 none
  TABLE fe_bar_lakebase.fraud_gov.txn_features             none

Genie space permissions:
  alex.owen@databricks.com     [('CAN_MANAGE', 'inherited')]
  admins  